# Classifying variants with a large language model (few-shot prompting)

Here we use a **generative large language model** (LLM), like the ones behind chatbots. It can follow written instructions, so we can:

1. **describe** the categories in plain words (the *instructions*);
2. show it a few **examples** classified by us (**few-shot**: a few examples; **zero-shot**: no examples);
3. ask it to classify a new variant and to **explain** its choice.

Then we compare its answers with our own classification (the *ground truth*).

The model is too big to run on our computers or on myBinder. It runs on a server of the University of Bern, **GPUStack**: we send our questions to it over the internet, through an **API** (*Application Programming Interface*: a way for programs to talk to each other).

## Connect to the university's model

We need three things:
- `API_URL`: the address of the server;
- `MODEL_NAME`: the name of the model on that server;
- the **API key**: a personal password that gives access to the server.

Run the cell below and paste the key in the box.

Attention! The key is like a password: do not write it in the notebook and do not share it. Create a separate .env file for it.

In [1]:
import requests  # Sends requests over the internet.
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
MODEL_NAME = os.getenv("MODEL_NAME")
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",  # The key, sent with every request.
    "Content-Type": "application/json",    # We send our request in JSON format.
}

## The data: our ground truth

The **ground truth** is the set of variants we classified by hand. We test the model on these variants only, because for them we know "our" answer and can check the model's.

The file needs two columns: `mod_id` (the identifier of the variant) and `category` (our category). Write its name in `GT_FILE`.

The readings and their context are taken from `../session06/variants_features.csv`, using `mod_id`.

In [1]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd  # Provides DataFrames for tables.
pd.set_option("display.max_colwidth", 120)  # Shows longer texts in table cells.

GT_FILE = "gt_fake.csv"  # The file with our classification.
CATEGORIES = ["error", "substantive", "vocabulary"]  # The categories decided in class.

ours = pd.read_csv(GT_FILE)[["mod_id", "category"]]
ours["category"] = ours["category"].str.strip().str.lower()  # "Error " -> "error"
ours = ours[ours["category"].isin(CATEGORIES)]  # Drops empty or unknown categories.

# Adds the readings and their context, taken from the session 6 table.
variants = pd.read_csv(Path("../session06/variants_features.csv"))
ground_truth = variants.merge(ours, on="mod_id")

print(len(ground_truth), "variants in the ground truth")
ground_truth["category"].value_counts()

60 variants in the ground truth


category
vocabulary     20
error          20
substantive    20
Name: count, dtype: int64

## Ground truth, examples and test set

We split the ground truth in two groups:
- **examples**: a few variants per category, shown to the model in the instructions;
- **test set**: all the others. The model classifies them, and we check its answers against ours.

A variant used as an example must not be in the test set, otherwise we would be giving the model the answer.

In [3]:
N_EXAMPLES = 2  # Examples per category. 0 = zero-shot.

examples = ground_truth.groupby("category").sample(N_EXAMPLES, random_state=1) if N_EXAMPLES else ground_truth.iloc[:0]
test = ground_truth.drop(examples.index)

print(len(examples), "examples,", len(test), "variants in the test set")
examples[["del_full", "add_full", "category"]]

6 examples, 54 variants in the test set


,del_full,add_full,category
15,aboul d,about,error
54,breehe,breeze,error
49,two leagues,half a league mile,substantive
28,season,winter,substantive
10,blossomed,bloomed,vocabulary
6,smallness,minuteness,vocabulary


## The instructions (the *prompt*)

The model only knows what we write here. The definitions are the ones we discussed in class.

This is the part to experiment with: change the definitions and see how the results change.

In [4]:
DEFINITIONS = """\
- error: a mistake is corrected: a typo, a misspelling, a grammatical problem.
- substantive: the meaning of the text changes (different content, facts, names, events).
- vocabulary: same meaning, better wording: a synonym, a more precise, effective or elegant word."""

def show_variant(row):
    # Describes one variant as text: the passage before and after, and the replaced words.
    return (f'Before: "{row["del_in_context"]}"\n'
            f'After:  "{row["add_in_context"]}"\n'
            f'Replaced: "{row["del_full"]}" -> "{row["add_full"]}"')

instructions = f"""You are an expert in textual scholarship. You classify revisions in the manuscript of Mary Shelley's Frankenstein.
Each revision replaces some words with others. Classify it in one of these categories:
{DEFINITIONS}

Answer in this format: category | one short sentence explaining why."""

if len(examples):
    instructions += "\n\nExamples:\n" + "\n".join(
        f'"{r["del_full"]}" -> "{r["add_full"]}": {r["category"]}' for _, r in examples.iterrows())

print(instructions)

You are an expert in textual scholarship. You classify revisions in the manuscript of Mary Shelley's Frankenstein.
Each revision replaces some words with others. Classify it in one of these categories:
- error: a mistake is corrected: a typo, a misspelling, a grammatical problem.
- substantive: the meaning of the text changes (different content, facts, names, events).
- vocabulary: same meaning, better wording: a synonym, a more precise, effective or elegant word.

Answer in this format: category | one short sentence explaining why.

Examples:
"aboul d" -> "about": error
"breehe" -> "breeze": error
"two leagues" -> "half a league mile": substantive
"season" -> "winter": substantive
"blossomed" -> "bloomed": vocabulary
"smallness" -> "minuteness": vocabulary


## Ask the model about one variant

A request to the model contains:
- `messages`, in two parts:
  - `system`: the instructions, the same for every variant;
  - `user`: the variant to classify;
- `temperature=0`: we ask for the most likely answer, so that the results are (almost) the same each time;
- `max_tokens`: the maximum length of the answer.

The server sends back a JSON answer; the text of the model is in `["choices"][0]["message"]["content"]`.

Some models "think aloud" before answering, between `<think>` and `</think>`. We remove that part and keep only the answer.

In [5]:
models_url = API_URL.replace("/chat/completions", "/models")
print(requests.get(models_url, headers=HEADERS, timeout=60).text)

{"data":[{"id":"jina-reranker-v2-base-multilingual","created":1775544997,"object":"model","owned_by":"gpustack","meta":null},{"id":"qwen3.8-27b","created":1787238154,"object":"model","owned_by":"gpustack","meta":null},{"id":"qwen3-vl-30b-a3b-instruct","created":1791194203,"object":"model","owned_by":"gpustack","meta":null},{"id":"faster-whisper-large-v3","created":1773974412,"object":"model","owned_by":"gpustack","meta":null},{"id":"gpt-oss-120b","created":1773974412,"object":"model","owned_by":"gpustack","meta":null},{"id":"internvl3-8b-instruct","created":1773974412,"object":"model","owned_by":"gpustack","meta":null},{"id":"granite-embedding-107m-multilingual","created":1773974412,"object":"model","owned_by":"gpustack","meta":null},{"id":"qwen3-embedding-4b","created":1791194370,"object":"model","owned_by":"gpustack","meta":null},{"id":"lfm2.5-24b-a2b","created":1792751061,"object":"model","owned_by":"gpustack","meta":null},{"id":"minimax-m2.7","created":1778839831,"object":"model","

In [6]:
# 1. The exact values: repr() also shows hidden spaces or line breaks.
print("URL:  ", repr(API_URL))
print("Model:", repr(MODEL_NAME))

# 2. Is the model in the server's list?
models_url = API_URL.replace("/chat/completions", "/models")
ids = [m["id"] for m in requests.get(models_url, headers=HEADERS, timeout=60).json()["data"]]
print("Model found on the server:", MODEL_NAME in ids)

# 3. The simplest possible request.
payload = {"model": MODEL_NAME, "messages": [{"role": "user", "content": "Say hello."}]}
response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
print("Simple request:", response.status_code, response.text[:500])

# 4. The same request as the notebook: with a system message, temperature and max_tokens.
payload = {"model": MODEL_NAME,
           "messages": [{"role": "system", "content": "You are helpful."},
                        {"role": "user", "content": "Say hello."}],
           "temperature": 0, "max_tokens": 1000}
response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
print("Notebook-style request:", response.status_code, response.text[:500])

URL:   'https://gpustack.unibe.ch/v1/chat/completions'
Model: 'qwen3.8-27b'
Model found on the server: True
Simple request: 200 {"id":"chatcmpl-ccffe9a8-7989-43ac-ae8f-a8f3b2edcaf8","object":"chat.completion","created":1793310164,"model":"qwen3.8-27b","choices":[{"index":0,"message":{"role":"assistant","content":"\n\nHello!","refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[],"reasoning":"We need to respond to user: \"Say hello.\" Simple. Just output Hello likely. Need final concise.\n"},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null}],"service_tier":null,"syste
Notebook-style request: 200 {"id":"chatcmpl-b243b9b3-b011-4bb0-accf-44bb42492b8d","object":"chat.completion","created":1793310165,"model":"qwen3.8-27b","choices":[{"index":0,"message":{"role":"assistant","content":"\n\nHello!","refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[],"reasoning":"We need to respond to user: \"Say hello.\" Simp

In [7]:
import re  # Regular expressions, to remove the "thinking" part.
import time  # To wait between requests.

def ask(row):
    # Sends one variant to the model and returns its answer (a text).
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "system", "content": instructions},
                     {"role": "user", "content": show_variant(row)}],
        "temperature": 0,
        "max_tokens": 1000,
    }
    for attempt in range(5):
        try:
            response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
        except requests.exceptions.RequestException as error:  # No connection, or the server is too slow.
            print("Connection problem, waiting 20 seconds:", str(error)[:100])
            time.sleep(20)
            continue
        if response.status_code == 429 or response.status_code >= 500:  # Too many requests, or server busy.
            print("Server busy, waiting 20 seconds:", response.status_code)
            time.sleep(20)
            continue
        response.raise_for_status()  # Other problems (e.g. wrong key or URL): stop and show the error.
        answer = response.json()["choices"][0]["message"]["content"] or ""
        return re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return ""

def read_category(answer):
    # Extracts the category from an answer such as "vocabulary | a synonym of ...".
    first_part = answer.split("|")[0].lower()
    for category in CATEGORIES:
        if category in first_part:
            return category
    return "unknown"

row = test.iloc[0]
print(show_variant(row))
answer = ask(row)
print("\nModel:", answer)
print("Category:", read_category(answer), "| Ours:", row["category"])

Before: "I cannot help here remark ing the many opportunities instructors have of directing the attention of their pupils to useful knowledge,"
After:  "I cannot help here remark ing the many opportunities instructors posess of directing the attention of their pupils to useful knowledge,"
Replaced: "have" -> "posess"

Model: vocabulary | “possess” (misspelled as “posess”) is a more formal synonym for “have” with essentially the same meaning.
Category: vocabulary | Ours: vocabulary


## Classify the test set

One request per variant: this takes a few seconds per variant.

In [8]:
answers = []
for i, (_, row) in enumerate(test.iterrows(), start=1):
    answers.append(ask(row))
    print(i, "/", len(test), end="\r")  # Progress.

test = test.assign(answer=answers)
test["predicted"] = test["answer"].apply(read_category)
test["reason"] = test["answer"].str.split("|").str[-1].str.strip()

test[["del_full", "add_full", "category", "predicted", "reason"]]

,del_full,add_full,category,predicted,reason
0,have,posess,vocabulary,unknown,
1,philosopheres,philosophers,error,error,The revision corrects a misspelling of “philosophers.”
2,Carignan,Clerval,substantive,substantive,"the revision changes a character's name, altering the text's content."
3,intended,intending,error,error,"changing ""intended"" to ""intending"" corrects the verb form to make the participial phrase grammatical."
4,chemistry,natural philosophy,substantive,substantive,It changes the specific field Victor despises from chemistry to the broader discipline of natural philosophy.
5,machines,instruments,vocabulary,vocabulary,"""instruments"" is a more precise and elegant synonym for ""machines"" without changing the meaning."
7,a helpmate,the accomplice,substantive,vocabulary,"""accomplice"" is a more precise and forceful synonym for ""helpmate"" in the context of vices."
8,punish,chastise,vocabulary,vocabulary,"""chastise"" is a more precise and elevated synonym for ""punish"" with essentially the same meaning."
9,any,the authors of any,substantive,substantive,The revision changes what is being studied from “any other country” to “the authors of any other country.”
11,sheweng,shewing,error,error,The revision corrects a misspelling of “shewing.”


## Evaluation

- **Accuracy**: share of variants where the model agrees with us.
- **Confusion matrix**: rows = our category, columns = the model's category. The diagonal holds the agreements; the other cells show *which* categories are confused.

In [9]:
accuracy = (test["predicted"] == test["category"]).mean()
print(f"Accuracy: {accuracy:.0%} on {len(test)} variants")
pd.crosstab(test["category"], test["predicted"], rownames=["ours"], colnames=["model"])

Accuracy: 87% on 54 variants


model,error,substantive,unknown,vocabulary
ours,,,,
error,17,0,1,0
substantive,0,13,2,3
vocabulary,0,0,1,17


In [10]:
# The disagreements, with the model's explanation: who is right?
test.loc[test["predicted"] != test["category"], ["del_full", "add_full", "category", "predicted", "reason"]]

,del_full,add_full,category,predicted,reason
0,have,posess,vocabulary,unknown,
7,a helpmate,the accomplice,substantive,vocabulary,"""accomplice"" is a more precise and forceful synonym for ""helpmate"" in the context of vices."
13,world,"life,",substantive,unknown,
31,determined,determining,error,unknown,
42,thought,appearance,substantive,unknown,
43,this,my feelings,substantive,vocabulary,"The revision replaces a pronoun with a more explicit phrase referring to the same concealed emotion, improving clari..."
51,situation,solitude,substantive,vocabulary,"""solitude"" is a more precise and evocative word for the same isolated circumstance."


## Save to CSV file

In [11]:
# Saves the classified variants, with the model's answers, to a CSV file.
# The model name is part of the file name, so results of different models are kept separate.
OUTPUT_FILE = "variants_classified_llm_" + re.sub(r"[^A-Za-z0-9.-]", "_", MODEL_NAME) + ".csv"  # "/" or ":" are not allowed in file names.
test.to_csv(OUTPUT_FILE, index=False)
print("Saved to", OUTPUT_FILE)

Saved to variants_classified_llm_qwen3.8-27b.csv


## Things to keep in mind

- **Plausible ≠ correct.** The explanations sound convincing even when the category is wrong. Read them critically.
- **Not fully reproducible.** The same prompt can give different answers on different days, and the models offered on a server change or disappear. Always note which model you used.
- **The prompt is part of the method.** Small changes in the definitions change the results: document the prompt you used.
- **Where does the data go?** Here, to a university server. Check where a model runs before sending confidential or personal data.
- **Ground truth is a choice too.** Another person might classify some variants differently: the "right" answer is sometimes debatable.

## Exercises

1. Set `N_EXAMPLES = 0` (zero-shot) and run sections 2–6 again. Does accuracy change? Then try `N_EXAMPLES = 4`.
2. Rewrite `DEFINITIONS` (section 3), for example with more precise criteria for *vocabulary* vs *substantive*. Does accuracy improve?
3. Look at the disagreements (section 6). Are there cases where you now agree with the model rather than with the ground truth?